In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [2]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lead, date_sub, coalesce, date_format
from pyspark.sql.window import Window

In [3]:
spark = get_sparkSession("Load gold.fact_orders")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [6]:
_schema_name = "gold"
_table_name = "fact_orders"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_fact_orders.ipynb"
_silver_table = "silver.fact_orders"
_bad_table = f"bad_{_table_name}"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for gold.fact_orders


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from silver.customer and generating surrogate key and business key

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)

SPARK-APP: Silver Table Data count : 2
+--------+--------+----------+----------+-----------+------+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+--------------------------+------------------------+--------------------------+------------------------+---------------+
|order_id|line_num|order_date|ship_date |customer_id|store |channel|product_id|currency|quantity|unit_price|list_price|shipping_amount|shipping_paid_by_customer|tax_amount|order_status|payment_status|created_on                |created_by              |modified_on               |modified_by             |source_file    |
+--------+--------+----------+----------+-----------+------+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+--------------------------+------------------------+--------------------------+------------------------+---------------

In [9]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

SPARK-APP: DQ validations completed
SPARK-APP: Table Data count after DQ validations : 2


In [10]:
#Adding audit columns

df = df.withColumn("created_on", current_timestamp()) \
       .withColumn("created_by", lit(_script_name)) \
       .withColumn("modified_on", current_timestamp()) \
       .withColumn("modified_by", lit(_script_name))

print(f"SPARK-APP: Silver Table Data count : {df.count()}")

# Generating surrogate key and business key

df_sk = df.withColumn("order_sk", xxhash64(concat_ws("||", "order_id", "line_num")).cast("bigint")) \
          .withColumn("customer_bk", xxhash64(concat_ws("||", "customer_id", "store", "channel")).cast("bigint"))

df_sk.show(truncate = False)


SPARK-APP: Silver Table Data count : 2
+--------+--------+----------+----------+-----------+------+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+-------------------------+----------------------+-------------------------+----------------------+---------------+-------------------+--------------------+
|order_id|line_num|order_date|ship_date |customer_id|store |channel|product_id|currency|quantity|unit_price|list_price|shipping_amount|shipping_paid_by_customer|tax_amount|order_status|payment_status|created_on               |created_by            |modified_on              |modified_by           |source_file    |order_sk           |customer_bk         |
+--------+--------+----------+----------+-----------+------+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+-------------------------+----------------------+

In [11]:
# Join with other dimension tables to get their surrogate keys

df_gold_store = spark.read.table("gold.dim_store").select("store_sk", "store_code")
df_gold_channel = spark.read.table("gold.dim_channel").select("channel_sk", "channel_code")
df_order_date = spark.read.table("gold.dim_date").select("date_sk","date")
df_ship_date = spark.read.table("gold.dim_date").select("date_sk","date")
df_gold_customer = spark.read.table("gold.dim_customer").where("is_current = true").select("customer_sk", "customer_bk")
df_gold_currency = spark.read.table("gold.dim_currency").select("currency_sk", "currency_code")

df_invalid = df_sk.join(df_gold_customer, how = 'left_anti', on=df_sk.customer_bk==df_gold_customer.customer_bk)

print(f"SPARK-APP: Bad Record Count : {df_invalid.count()}")  

df_invalid.show(truncate = False)

if df_invalid.count() > 0:
    df_invalid.withColumn('error_description', lit("Customer not found in gold.dim_customer")) \
                       .select("order_id", "line_num", "order_date", "ship_date", "customer_id", "product_id", "store", "channel", "currency", "quantity", 
                               "unit_price", "list_price", "shipping_amount", "shipping_paid_by_customer", "tax_amount", "order_status", "payment_status", 
                               "source_file", "created_on", "created_by", "modified_on", "modified_by", "error_description") \
                       .write \
                       .format("delta") \
                       .mode("append") \
                       .saveAsTable(f"bad.{_bad_table}")

df_silver = df_sk.join(df_gold_store, how="inner", on=df_sk.store==df_gold_store.store_code) \
                 .join(df_gold_channel, how="inner", on=df_sk.channel==df_gold_channel.channel_code) \
                 .join(df_gold_customer, how = "inner", on=df_sk.customer_bk==df_gold_customer.customer_bk) \
                 .join(df_gold_currency, how = "inner", on=df_sk.currency==df_gold_currency.currency_code) \
                 .join(df_order_date, how = "left", on=df_sk.order_date==df_order_date.date) \
                 .join(df_ship_date, how = "left", on=df_sk.ship_date==df_ship_date.date) \
                 .select("order_sk", "order_id", "line_num", coalesce(df_order_date.date_sk,date_format(df_sk.order_date, "yyyyMMdd").cast("int")).alias("order_date"), 
                         coalesce(df_ship_date.date_sk, date_format(df_sk.ship_date, "yyyyMmdd").cast("int")).alias("ship_date"), "customer_sk", 
                         "product_id", "store_sk", "channel_sk", "currency_sk", "quantity", "unit_price", "list_price", "shipping_amount", "shipping_paid_by_customer", 
                         "tax_amount", "order_status", "payment_status", "source_file", "created_on", "created_by", "modified_on", "modified_by")

df_silver.show(truncate = False)

SPARK-APP: Bad Record Count : 0
+--------+--------+----------+---------+-----------+-----+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+----------+----------+-----------+-----------+-----------+--------+-----------+
|order_id|line_num|order_date|ship_date|customer_id|store|channel|product_id|currency|quantity|unit_price|list_price|shipping_amount|shipping_paid_by_customer|tax_amount|order_status|payment_status|created_on|created_by|modified_on|modified_by|source_file|order_sk|customer_bk|
+--------+--------+----------+---------+-----------+-----+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+----------+----------+-----------+-----------+-----------+--------+-----------+
+--------+--------+----------+---------+-----------+-----+-------+----------+--------+--------+----------+----------+---------------+-

In [12]:
#Derive additional columns

df_silver = df_silver.withColumn("total_price", (col("unit_price") * col("quantity")).cast("decimal(18,2)")) \
                     .withColumn("discount_amount", ((col("list_price") - col("unit_price")) * col("quantity")).cast("decimal(18,2)")) \
                     .withColumn("gross_revenue", (when(col("shipping_paid_by_customer")=="true", col("total_price") + col("shipping_amount")).otherwise(col("total_price"))).cast("decimal(18,2)")) \
                     .withColumn("net_revenue", when(col("shipping_paid_by_customer")=="true", col("total_price")).otherwise(col("total_price") - col("shipping_amount")).cast("decimal(18,2)")) 

df_silver.printSchema()
df_silver.show()

root
 |-- order_sk: long (nullable = false)
 |-- order_id: string (nullable = true)
 |-- line_num: integer (nullable = true)
 |-- order_date: integer (nullable = true)
 |-- ship_date: integer (nullable = true)
 |-- customer_sk: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_sk: integer (nullable = true)
 |-- channel_sk: integer (nullable = true)
 |-- currency_sk: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- list_price: decimal(18,2) (nullable = true)
 |-- shipping_amount: decimal(18,2) (nullable = true)
 |-- shipping_paid_by_customer: boolean (nullable = true)
 |-- tax_amount: decimal(18,2) (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- created_on: timestamp (nullable = false)
 |-- created_by: string (nullable = false)
 |-- modified_on: timestamp (nullable = false)
 |-- modif

In [13]:
# Truncate table for full load
dt_fact_orders = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_fact_orders.delete()
    dt_fact_orders.vacuum(0)


In [14]:
# Merge
dt_fact_orders.alias("t").merge(
    df_silver.alias("s"),
    "t.order_sk = s.order_sk"
).whenMatchedUpdate(set = {   
    "quantity":"s.quantity",
    "unit_price":"s.unit_price",
    "list_price":"s.list_price",
    "shipping_amount":"s.shipping_amount",
    "shipping_paid_by_customer":"s.shipping_paid_by_customer",
    "total_price":"s.total_price",
    "discount_amount":"s.discount_amount",
    "gross_revenue":"s.gross_revenue",
    "net_revenue":"s.net_revenue",
    "tax_amount":"s.tax_amount",
    "order_status":"s.order_status",
    "payment_status":"s.payment_status",
    "source_file" : "s.source_file",
    "modified_on" : "s.modified_on",
    "modified_by" : "s.modified_by"
    }).whenNotMatchedInsertAll().execute()
                     
print("SPARK-APP: Merge completed") 

SPARK-APP: Merge completed


In [15]:
hist = dt_fact_orders.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|      1|           6973|                    2|                   0|            2|
+-------+---------------+---------------------+--------------------+-------------+



In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+-------------------+--------+--------+----------+---------+-------------------+----------+--------+----------+-----------+--------+----------+----------+---------------+-------------------------+-----------+---------------+-------------+-----------+----------+------------+--------------+--------------------+--------------------+--------------------+--------------------+---------------+
|           order_sk|order_id|line_num|order_date|ship_date|        customer_sk|product_id|store_sk|channel_sk|currency_sk|quantity|unit_price|list_price|shipping_amount|shipping_paid_by_customer|total_price|discount_amount|gross_revenue|net_revenue|tax_amount|order_status|payment_status|          created_on|          created_by|         modified_on|         modified_by|    source_file|
+-------------------+--------+--------+----------+---------+-------------------+----------+--------+----------+-----------+--------+----------+----------+---------------+-------------------------+-----------+------------

In [17]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

SPARK-APP: Data written to load_controller


In [18]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+-----------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|layer|schema_name| table_name| load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+-----+-----------+-----------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| gold|       gold|fact_orders|delta-load|     merge|2026-01-29 03:01:...|    success|           2|2026-01-29 03:01:...|gold_fact_orders....|2026-01-29 03:01:...|gold_fact_orders....|
+-----+-----------+-----------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [19]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

SPARK-APP: symlink manifest file generated


In [20]:
spark.stop()